# Cleaning

## Merging all files

In [ ]:
import pandas as pd
from pathlib import Path


files = sorted(Path("synop").glob("*.csv"))

dfs = []

for file in files:
    df = pd.read_csv(file, sep=";")
    df["source_file"] = file.name
    dfs.append(df)

synop = pd.concat(dfs, ignore_index=True)

## Duplicate

In [ ]:
float_cols = ["t", "u", "ff", "dd", "pmer", "rr3"]
synop[float_cols] = synop[float_cols].astype("float32")

synop=synop.drop_duplicates()

synop = synop.drop_duplicates(
    subset=["geo_id_wmo", "validity_time"],
    keep="last"
)


## Fill

In [ ]:
cols = ["t", "u", "ff", "pmer"]

synop = synop.sort_values(["geo_id_wmo", "validity_time"])

synop[cols] = (
    synop.groupby("geo_id_wmo")[cols]
      .transform(lambda x: x.interpolate(method="linear", limit_direction="both"))
)





In [ ]:
for col in ["t", "u", "ff", "pmer"]:
    synop[col] = synop[col].fillna(
        synop.groupby("geo_id_wmo")[col].transform("mean")
    )
for col in ["t", "u", "ff", "pmer"]:
    synop[col] = synop[col].fillna(synop[col].mean())

In [ ]:
synop["dd"] = (
    synop.groupby("geo_id_wmo")["dd"]
      .transform(lambda x: x.ffill().bfill())
)

## Outliers

In [ ]:
cols = ["t", "u", "ff", "pmer"]

mask = pd.Series(False,index=synop.index)

for col in cols:
    Q1 = synop.groupby("geo_id_wmo")[col].transform(
        lambda x: x.quantile(0.25)
    )

    Q3 = synop.groupby("geo_id_wmo")[col].transform(
        lambda x: x.quantile(0.75)
    )

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    mask|= ~synop[col].between(lower,upper)
    synop[col + "_outlier"] = ~synop[col].between(lower, upper)

synop_clean = synop[~mask]

In [ ]:
cols = ["t", "u", "ff", "pmer"]

for col in cols:
    bounds = synop_clean.groupby("geo_id_wmo")[col].agg(
        Q1=lambda x: x.quantile(0.25),
        Q3=lambda x: x.quantile(0.75)
    )
    bounds["IQR"] = bounds["Q3"] - bounds["Q1"]
    bounds["lower"] = bounds["Q1"] - 1.5*bounds["IQR"]
    bounds["upper"] = bounds["Q3"] + 1.5*bounds["IQR"]

    
    df = synop_clean.merge(bounds[["lower","upper"]], left_on="geo_id_wmo", right_index=True)
    
    synop_clean[col] = df[col].clip(lower=df["lower"], upper=df["upper"])

In [ ]:
cols = ["t", "u", "ff", "pmer"]
outliers_iqr = {}

for col in cols:
    
    bounds = synop_clean.groupby("geo_id_wmo")[col].agg(
        Q1=lambda x: x.quantile(0.25),
        Q3=lambda x: x.quantile(0.75)
    )
    bounds["IQR"] = bounds["Q3"] - bounds["Q1"]
    bounds["lower"] = bounds["Q1"] - 1.5*bounds["IQR"]
    bounds["upper"] = bounds["Q3"] + 1.5*bounds["IQR"]

    
    df = synop_clean.merge(bounds[["lower","upper"]], left_on="geo_id_wmo", right_index=True)

   
    mask = (df[col] < df["lower"]) | (df[col] > df["upper"])
    outliers_iqr[col] = df[mask][col]
    print(f"{col}: {len(outliers_iqr[col])} outliers detected")

## End

In [ ]:
final_synop = synop_clean[
    [
        "geo_id_wmo",
        "geo_id_wigos",
        "name",
        "lat",
        "lon",
        "validity_time",
        "t",
        "u",
        "ff",
        "dd",
        "pmer",
        "rr3"
    ]
]

In [ ]:
final_synop = final_synop.copy()

final_synop["rr3"] = (
    pd.to_numeric(final_synop["rr3"], errors="coerce")
    .fillna(0)
)
final_synop.to_csv("Synop-cleaned/clean_synop.csv",index=False)

In [ ]:
final_synop.shape